# Regression with LightGBM V3.1 (oracle) — Income, Gradient-Boosted Trees, k=5

**Version:** V3.1 (k-fold validation — LightGBM ceiling check). **Role:** reference oracle for the
V3.1 scratch twin. Same `LGBMRegressor` hypers as `regression_lightgbm_V3_0.ipynb` (MSE objective,
500-round cap, 50-round early stopping on a 10%-of-train slice), copied verbatim and wrapped in a
5-fold `KFold` loop over the full 2,212-row dropped-target set instead of one fixed 80/20 split.
Weights → `./weights_V3_1/`.

In [1]:
import json
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
os.makedirs("weights_V3_1", exist_ok=True)
print("lightgbm:", lgb.__version__)

df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()
print("cleaned:", df.shape, "| Income median:", round(df["Income"].median()))

REG_FEATURES = [
    "NumCatalogPurchases", "MntMeatProducts", "MntWines", "NumWebVisitsMonth",
    "MntSweetProducts", "MntFruits", "NumWebPurchases", "MntGoldProds",
    "NumStorePurchases", "MntFishProducts", "Kidhome", "Age", "NumDealsPurchases",
]
X_full = df[REG_FEATURES].values
y_full = df["Income"].astype(float).values
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

lightgbm: 4.7.0
cleaned: (2212, 28) | Income median: 51371
features: 13 dims (all numeric)


## K-fold training (k=5) — fresh LGBMRegressor per fold, own 10%-of-train early-stopping slice

In [2]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_full), start=1):
    Xtr_f, Xva_f = X_full[tr_idx], X_full[va_idx]
    ytr_f, yva_f = y_full[tr_idx], y_full[va_idx]

    X_tr, X_val, y_tr, y_val = train_test_split(Xtr_f, ytr_f, test_size=0.10, random_state=SEED)

    reg = lgb.LGBMRegressor(
        objective="regression", n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, verbosity=-1,
    )
    reg.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], eval_names=["train", "valid"],
            eval_metric="rmse", callbacks=[lgb.early_stopping(50, verbose=False)])

    pred_v = reg.predict(Xva_f)
    mse_v = float(mean_squared_error(yva_f, pred_v))
    fold_rows.append({
        "fold": fold,
        "best_iteration": int(reg.best_iteration_),
        "test_loss_MSE_raw": round(mse_v, 1),
        "rmse": round(float(np.sqrt(mse_v)), 1),
        "mae": round(float(mean_absolute_error(yva_f, pred_v)), 1),
        "r2": round(float(r2_score(yva_f, pred_v)), 4),
    })
    print(f"fold {fold}: RMSE={fold_rows[-1]['rmse']:.1f} R2={fold_rows[-1]['r2']:.4f} best_iter={fold_rows[-1]['best_iteration']}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_loss_MSE_raw", "rmse", "mae", "r2")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V3_1/regression_lightgbm_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 1: RMSE=8582.0 R2=0.8449 best_iter=305


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 2: RMSE=10063.7 R2=0.7733 best_iter=499


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 3: RMSE=7229.2 R2=0.8834 best_iter=500


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 4: RMSE=10001.9 R2=0.7970 best_iter=500


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 5: RMSE=8698.1 R2=0.8315 best_iter=329
 fold  best_iteration  test_loss_MSE_raw    rmse    mae     r2
    1             305         73650847.6  8582.0 5485.2 0.8449
    2             499        101278343.9 10063.7 5318.8 0.7733
    3             500         52261610.8  7229.2 5226.7 0.8834
    4             500        100038244.7 10001.9 5972.6 0.7970
    5             329         75657054.6  8698.1 5655.1 0.8315
{
  "test_loss_MSE_raw": {
    "mean": 80577220.32,
    "std": 20501368.2449
  },
  "rmse": {
    "mean": 8914.98,
    "std": 1172.6806
  },
  "mae": {
    "mean": 5531.68,
    "std": 295.8606
  },
  "r2": {
    "mean": 0.826,
    "std": 0.0427
  }
}
